In [1]:
import kagglehub

path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")
print(path) 


/opt/homebrew/Caskroom/miniconda/base/envs/apache_spark/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 653M/653M [01:07<00:00, 10.1MB/s] 

Extracting files...


/Users/mw/.cache/kagglehub/datasets/sobhanmoosavi/us-accidents/versions/13


In [6]:
!cp /Users/mw/.cache/kagglehub/datasets/sobhanmoosavi/us-accidents/versions/13/US_Accidents_March23.csv ../data/US_Accidents_March23.csv

In [10]:
path = "../data/"

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
   .appName("Colab Spark Example") \
   .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
   .getOrCreate()


In [11]:

inDF = spark.read.format("csv") \
 .option("sep", ",") \
 .option("inferSchema", "true") \
 .option("header", "true") \
 .load(path + "/US_Accidents_March23.csv")

inDF.printSchema()


root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- V

In [15]:
inDF.select("city").groupBy("city").count().orderBy("count", ascending=False)


DataFrame[city: string, count: bigint]

In [ ]:
inDF.select("city").groupBy("city").count().orderBy("count", ascending=False).show(50)

+-------------+------+
|         city| count|
+-------------+------+
|        Miami|186917|
|      Houston|169609|
|  Los Angeles|156491|
|    Charlotte|138652|
|       Dallas|130939|
|      Orlando|109733|
|       Austin| 97359|
|      Raleigh| 86079|
|    Nashville| 72930|
|  Baton Rouge| 71588|
|      Atlanta| 68186|
|   Sacramento| 66264|
|    San Diego| 55504|
|      Phoenix| 53974|
|  Minneapolis| 51488|
|     Richmond| 48845|
|Oklahoma City| 46092|
| Jacksonville| 42447|
|       Tucson| 39304|
|     Columbia| 38178|
+-------------+------+
only showing top 20 rows


In [16]:
# Filtrowanie tylko niepustych warunków pogodowych
filtered = inDF.filter(inDF["Weather_Condition"]=='Rain')


# CACHE – przechowujemy dane w RAM
filtered.cache()


26/06/27 10:24:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[ID: string, Source: string, Severity: int, Start_Time: timestamp, End_Time: timestamp, Start_Lat: double, Start_Lng: double, End_Lat: double, End_Lng: double, Distance(mi): double, Description: string, Street: string, City: string, County: string, State: string, Zipcode: string, Country: string, Timezone: string, Airport_Code: string, Weather_Timestamp: timestamp, Temperature(F): double, Wind_Chill(F): double, Humidity(%): double, Pressure(in): double, Visibility(mi): double, Wind_Direction: string, Wind_Speed(mph): double, Precipitation(in): double, Weather_Condition: string, Amenity: boolean, Bump: boolean, Crossing: boolean, Give_Way: boolean, Junction: boolean, No_Exit: boolean, Railway: boolean, Roundabout: boolean, Station: boolean, Stop: boolean, Traffic_Calming: boolean, Traffic_Signal: boolean, Turning_Loop: boolean, Sunrise_Sunset: string, Civil_Twilight: string, Nautical_Twilight: string, Astronomical_Twilight: string]

In [17]:
filtered.groupBy("Weather_Condition").count().show()


+-----------------+-----+
|Weather_Condition|count|
+-----------------+-----+
|             Rain|84331|
+-----------------+-----+



In [18]:
filtered.filter(filtered["Weather_Condition"] == "Rain").count()


84331

In [19]:
df_repartitioned = inDF.repartition(16)
print(f"Liczba partycji po repartition(): {df_repartitioned.rdd.getNumPartitions()}")


Liczba partycji po repartition(): 16


In [ ]:
df_coalesced = inDF.coalesce(16)
print(f"Liczba partycji po coalesce(): {df_coalesced.rdd.getNumPartitions()}")



Liczba partycji po coalesce(): 16
